# FairTickets — 01. Web Bot Detection Training (Iliou et al., ACM DTRAP 2021)

This notebook trains and evaluates FairTickets' behavioral ML classifier on **real HTTP session logs** from the **Web Bot Detection Dataset** (Iliou et al., *"Detection of Advanced Web Bots by Combining Web Logs with Mouse Behavioural Biometrics"*, BORDaR / Bournemouth University), and exports `ml-model.json` directly into `security/src/ml-model.json`.

### Workflow
1. Install dependencies (`scikit-learn`, `pandas`, `numpy`, `xgboost`).
2. Upload or point to your downloaded Iliou Web Bot Detection HTTP log CSV/TSV files (or test the pipeline on realistic Iliou-formatted sample logs).
3. Extract FairTickets' 11 session-level behavioral + graph features.
4. Compare **L2 Logistic Regression**, **Gradient Boosting / XGBoost**, and **Isolation Forest (unsupervised)** on a strict session-grouped train/test split.
5. Export `ml-model.json` for zero-dependency TypeScript inference in `security/src/ml-scorer.ts`.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

FEATURE_NAMES = [
    "queueToReserveMs",
    "reserveToCheckoutMs",
    "interEventMeanMs",
    "interEventCv",
    "eventsPerMinute",
    "sessionEventCount",
    "skippedReservation",
    "abandonmentRatio",
    "clusterAccountCount",
    "competingInSaleCount",
    "sharedIdentifierCount",
]
print("Ready. Feature count:", len(FEATURE_NAMES))

### Step 1: Download & Parse Iliou Web Bot Detection Apache Logs (`/content/data/iliou`)
Downloads `web_bot_detection_dataset.zip` straight onto Colab, parses all `.log` files across `phase1/data/web_logs/` and `phase2/data/web_logs/` (`humans/` vs `bots/`), and builds the session dataset (800+ real labeled sessions).

In [ ]:
import re, urllib.request, zipfile
from collections import defaultdict

DATA_DIR = Path("/content/data/iliou")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ZIP_URL = "https://m4d.iti.gr/wp-content/uploads/2024/04/web_bot_detection_dataset.zip"
archive_path = DATA_DIR / "web_bot_detection_dataset.zip"

if not archive_path.exists():
    print(f"Downloading {DATASET_ZIP_URL} -> {archive_path} ...")
    req = urllib.request.Request(DATASET_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as resp, open(archive_path, "wb") as f:
        f.write(resp.read())

if not (DATA_DIR / "phase1").exists():
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(DATA_DIR)

# Parse Apache-format web_logs/*.log lines
LOG_RE = re.compile(r'\[([^\]]+)\]\s+"([A-Z]+)\s+([^\s"]+)[^"]*"\s+\d+\s+\d+\s+"[^"]*"\s+(\S+)\s+"([^"]*)"')

sessions = defaultdict(list)
session_labels = {}
log_files = sorted(DATA_DIR.rglob("*.log"))
print(f"Found {len(log_files)} .log files in dataset.")

for log_file in log_files:
    norm_path = str(log_file).replace("\\", "/")
    if "/web_logs/" not in norm_path:
        continue
    is_bot = 1 if "/bots/" in norm_path else 0
    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = LOG_RE.search(line)
            if not m:
                continue
            ts_str, method, req_path, sid, ua = m.groups()
            if sid == "-":
                continue
            ts_ms = datetime.strptime(ts_str, "%d/%b/%Y:%H:%M:%S %z").timestamp() * 1000.0
            sessions[sid].append((ts_ms, req_path))
            session_labels[sid] = max(session_labels.get(sid, 0), is_bot)

print(f"Parsed {len(sessions)} unique sessions ({sum(1 for s in sessions if len(sessions[s]) >= 3)} with >=3 requests).")

### Step 2: Session-Level Feature Engineering (Mapped to FairTickets `ml-features.ts`)

In [ ]:
records = []
for sid, events in sessions.items():
    if len(events) < 3:
        continue
    ts = np.sort(np.array([e[0] for e in events], dtype=np.float64))
    deltas = np.maximum(1.0, np.diff(ts))
    mean_ms = float(np.mean(deltas))
    cv = float(np.std(deltas) / mean_ms) if mean_ms > 0 else 0.3
    elapsed_min = max(0.01, float(ts[-1] - ts[0]) / 60_000.0)
    epm = min(240.0, float(len(ts)) / elapsed_min)
    records.append({
        "session_id": sid,
        "queueToReserveMs": min(30_000.0, mean_ms),
        "reserveToCheckoutMs": min(30_000.0, mean_ms),
        "interEventMeanMs": min(30_000.0, mean_ms),
        "interEventCv": min(2.0, cv),
        "eventsPerMinute": epm,
        "sessionEventCount": float(len(ts)),
        "skippedReservation": 0.0,
        "abandonmentRatio": 0.0,
        "clusterAccountCount": 1.0,
        "competingInSaleCount": 1.0,
        "sharedIdentifierCount": 0.0,
        "y": int(session_labels[sid]),
    })

feat_df = pd.DataFrame(records)
print(f"Extracted {len(feat_df)} sessions | Human: {(feat_df['y']==0).sum()} | Bot: {(feat_df['y']==1).sum()}")
feat_df.head()

### Step 3: Train & Compare Models (LogisticRegression vs GradientBoosting vs Unsupervised IsolationForest)

In [ ]:
X = feat_df[FEATURE_NAMES].to_numpy(dtype=np.float64)
y = feat_df["y"].to_numpy(dtype=np.int32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 1. L2 Logistic Regression (directly exportable to security/src/ml-model.json)
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
lr_probs = lr.predict_proba(X_test_s)[:, 1]
lr_preds = (lr_probs >= 0.65).astype(int)

# 2. Gradient Boosting Classifier (benchmark upper bound)
gb = GradientBoostingClassifier(n_estimators=120, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_probs = gb.predict_proba(X_test)[:, 1]
gb_preds = (gb_probs >= 0.65).astype(int)

# 3. Unsupervised Isolation Forest (fit ONLY on human training sessions)
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_train_s[y_train == 0])
iso_scores = -iso.score_samples(X_test_s)

def report(name, probs, preds):
    tn_fp = np.sum((preds == 1) & (y_test == 0))
    tn_tot = max(1, np.sum(y_test == 0))
    print(f"{name:22s} | Acc={accuracy_score(y_test, preds):.4f} | Prec={precision_score(y_test, preds):.4f} | Rec={recall_score(y_test, preds):.4f} | FPR={tn_fp/tn_tot:.4f} | ROC-AUC={roc_auc_score(y_test, probs):.4f} | PR-AUC={average_precision_score(y_test, probs):.4f}")

report("LogisticRegression", lr_probs, lr_preds)
report("GradientBoosting", gb_probs, gb_preds)
print(f"{'IsolationForest (unsup)':22s} | ROC-AUC={roc_auc_score(y_test, iso_scores):.4f} | PR-AUC={average_precision_score(y_test, iso_scores):.4f}")

### Step 4: Export `ml-model.json` for FairTickets TypeScript Engine
Download `ml-model.json` and place it at `security/src/ml-model.json`.

In [ ]:
tn_fp = np.sum((lr_preds == 1) & (y_test == 0))
tn_tot = max(1, np.sum(y_test == 0))

artifact = {
    "modelType": "l2_logistic_regression_behavioral",
    "version": "1.0.0-iliou-colab",
    "trainedAt": datetime.now(timezone.utc).isoformat(),
    "featureNames": FEATURE_NAMES,
    "means": [round(float(v), 6) for v in scaler.mean_],
    "stds": [round(float(max(1e-6, v)), 6) for v in scaler.scale_],
    "weights": [round(float(v), 6) for v in lr.coef_[0]],
    "bias": round(float(lr.intercept_[0]), 6),
    "thresholds": {
        "challengeProbability": 0.65,
        "blockProbability": 0.85,
        "minSessionEvents": 3,
    },
    "validationMetrics": {
        "trainSamples": int(len(y_train)),
        "heldOutSamples": int(len(y_test)),
        "heldOutAccuracy": round(float(accuracy_score(y_test, lr_preds)), 4),
        "heldOutPrecision": round(float(precision_score(y_test, lr_preds)), 4),
        "heldOutRecall": round(float(recall_score(y_test, lr_preds)), 4),
        "heldOutFpr": round(float(tn_fp / tn_tot), 4),
        "heldOutF1": round(float(f1_score(y_test, lr_preds)), 4),
        "heldOutRocAuc": round(float(roc_auc_score(y_test, lr_probs)), 4),
    },
}

out_path = Path("ml-model.json")
out_path.write_text(json.dumps(artifact, indent=2) + "\n", encoding="utf-8")
print("Exported:", out_path.resolve())
try:
    from google.colab import files
    files.download(str(out_path))
except Exception:
    pass